# AgriTech · entrenamiento IP102 balanceado

Abre este cuaderno en Google Colab y elige **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU** antes de ejecutarlo. El cuaderno descarga los dos archivos compartidos por Alonso, reentrena un **candidato** ResNet‑152 y lo compara con el peso no balanceado en el mismo conjunto de prueba. No publica el candidato en el sitio automáticamente.

Necesitarás iniciar sesión para montar tu Google Drive y conservar el resultado. El ZIP ocupa aproximadamente 3,7 GB; verifica que Colab tenga al menos 8 GB libres.

In [ ]:
import shutil, subprocess, sys
from pathlib import Path
import torch

assert torch.cuda.is_available(), 'Activa GPU en el entorno de ejecución de Colab antes de continuar.'
assert shutil.disk_usage('/content').free >= 8_000_000_000, 'Se necesitan al menos 8 GB libres en /content.'
print('GPU:', torch.cuda.get_device_name(0))
print('Espacio libre (GB):', round(shutil.disk_usage('/content').free / 1e9, 1))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown>=5,<6', 'safetensors>=0.4'], check=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT = Path('/content/drive/MyDrive/AgriTech-IP102-resultado')
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Los resultados quedarán en:', OUTPUT)

In [ ]:
import gdown, hashlib
WORK = Path('/content/AgriTech')
if not WORK.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'feature/multicultivo-resnet', 'https://github.com/jjjoaquo/AgriTech.git', str(WORK)], check=True)
DATASET = Path('/content/ip102_balanced_stylegan2.zip')
BASELINE_PTH = Path('/content/best_resnet152_baseline.pth')
FILES = {
    DATASET: ('1T58SblSeBcNT5aqSrx7Ookw5WJTOWz_X', 3_698_825_898, '1728e1d3989b4f2a0bf5994567680a35fc5472968ec745ab35ba9307e9c1fcc5'),
    BASELINE_PTH: ('10kw73CxA1F9m_nelh78AUXKQ32xvF1h1', 701_581_674, 'a3cd065ff991c294db0f69e1f1535da85d152e565ad516bc6b7bc6114bee2312'),
}
for path, (file_id, expected_size, expected_sha256) in FILES.items():
    if not path.exists() or path.stat().st_size != expected_size:
        gdown.download(id=file_id, output=str(path), quiet=False)
    assert path.is_file() and path.stat().st_size == expected_size, f'Descarga incompleta: {path}'
    digest = hashlib.sha256()
    with path.open('rb') as source:
        for chunk in iter(lambda: source.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    assert digest.hexdigest() == expected_sha256, f'Archivo distinto al recibido: {path}'
print('Archivos de Alonso verificados por tamaño y SHA-256.')

In [ ]:
# Conversión segura: extrae solo model_state_dict del checkpoint; no modifica el archivo original.
BASELINE_SAFE = Path('/content/best_resnet152_baseline.safetensors')
if not BASELINE_SAFE.exists():
    subprocess.run([sys.executable, str(WORK / 'mvp/convert_resnet_checkpoint.py'), str(BASELINE_PTH), str(BASELINE_SAFE)], check=True)
assert BASELINE_SAFE.stat().st_size > 200_000_000

In [ ]:
# Puede tardar varias horas según la GPU. Si falta memoria, cambia --batch-size a 8.
command = [
    sys.executable, str(WORK / 'mvp/train_ip102_balanced.py'),
    '--dataset', str(DATASET), '--baseline', str(BASELINE_SAFE), '--output', str(OUTPUT),
    '--head-epochs', '2', '--fine-epochs', '4', '--batch-size', '16', '--workers', '2',
]
subprocess.run(command, check=True)

In [ ]:
import json
report = json.loads((OUTPUT / 'comparison.json').read_text(encoding='utf-8'))
assert report['status'] == 'completed'
for label in ('baseline_test', 'candidate_test'):
    result = report[label]
    print(label, 'F1 macro:', result['macro_f1'], 'accuracy:', result['accuracy'], 'balanced accuracy:', result['balanced_accuracy'])
print('¿Mejora F1 macro?', report['candidate_improves_macro_f1'])
print('Peso candidato:', OUTPUT / 'best_resnet152_balanced_candidate.safetensors')
print('Informe:', OUTPUT / 'comparison.json')

## Después del entrenamiento

Comparte `comparison.json` y `best_resnet152_balanced_candidate.safetensors`. Antes de instalar el candidato hay que revisar F1 macro, recall por clase, posibles duplicados visuales y fotografías nuevas de los cultivos objetivo. El peso anterior debe conservarse para poder revertir el cambio.